# harness-admission on Colab (T4)
Run top to bottom. Everything lands in Drive (`MyDrive/ha_home`), so after a disconnect just rerun cells 1-3 and the step you were on: finished work is skipped.

In [11]:
!nvidia-smi   # must show Tesla T4 (Runtime > Change runtime type > T4 GPU)

Mon Nov 16 20:59:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   76C    P0             36W /   72W |   18874MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
from google.colab import drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!mkdir -p /content/ha_home
!git clone https://github.com/Soumya080/harness-admission.git /content/ha 2>/dev/null || (cd /content/ha && git pull)
%cd /content/ha

MessageError: Error: credential propagation was unsuccessful

In [17]:
import os, sys
!rm -rf /content/ha
!git clone https://github.com/Soumya080/harness-admission-colab.git /content/ha
os.chdir('/content/ha'); sys.path.insert(0, '/content/ha')
os.environ['HA_HOME'] = '/content/ha_home'
os.environ['HA_DTYPE'] = 'half'
os.environ['HA_GPU_UTIL'] = '0.85'
!mkdir -p /content/ha_home
!ls /content/ha | head -20
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

Cloning into '/content/ha'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 56 (delta 0), reused 56 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 73.43 KiB | 14.69 MiB/s, done.
analysis
colab_run.ipynb
data
fake_test.py
fig.png
gates
harness
proposer
README.md
requirements.txt
runner
scripts
sim
sims
tests
name, memory.used [MiB], memory.total [MiB]
NVIDIA L4, 18874 MiB, 23034 MiB


In [18]:
!pip install -q vllm datasets pandas pyarrow scipy matplotlib pytest
import os
os.environ['HA_HOME'] = '/content/drive/MyDrive/ha_home'
os.environ['HA_DTYPE'] = 'half'      # T4 has no bf16
os.environ['HA_GPU_UTIL'] = '0.85'

In [19]:
import os
os.environ['HA_HOME'] = '/content/drive/MyDrive/ha_home'
os.environ['HA_DTYPE'] = 'half'      # T4 has no bf16
os.environ['HA_GPU_UTIL'] = '0.85'

In [20]:
import torch, vllm
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("vllm", vllm.__version__)

torch 2.14.0+cu130 | CUDA True NVIDIA L4
vllm 0.33.0


In [ ]:
!python -m data.pools
!python -m proposer.manual
!python -m scripts.run_grid --only base
!python -m scripts.run_grid --only planted
!python -m scripts.check_stop

In [23]:
!pkill -9 -f EngineCore; pkill -9 -f "scripts.run_grid"; pkill -9 -f "proposer.propose"; pkill -9 -f vllm; sleep 5
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

(MainProcess pid=15471) INFO 11-16 21:11:18 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore
^C
name, memory.used [MiB], memory.total [MiB]
NVIDIA L4, 18874 MiB, 23034 MiB


In [24]:
import subprocess
used = int(subprocess.check_output("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits", shell=True).split()[0])
assert used < 1000, f"GPU still has {used} MiB in use: kill the leftover process first"
!python -m proposer.manual
!python -m scripts.run_grid --only base && python -m scripts.run_grid --only planted && python -m scripts.check_stop

base     5d55c5af1e
planted  467de61d65
noop     25641245b0
noop     b2e3569ada
noop     40a4bd62d0
noop     f94fd3d991
noop     ee3ede2a72
1 harnesses
W1116 21:11:36.207000 27626 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1116 21:11:36.207000 27626 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1116 21:11:36.208000 27626 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=27626) INFO 11-16 21:11:44 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
(MainProcess pid=27626) INFO 11-16 21:11:45 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=27626) WARNING 11-16 21:11:45 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=27626) INFO 11-16 21:11:45 [model.p

In [25]:
import os
os.environ['HA_DTYPE'] = 'half'
from vllm import LLM, SamplingParams
llm = LLM("Qwen/Qwen2.5-3B-Instruct", dtype="half", max_model_len=2048, gpu_memory_utilization=0.85)
out = llm.chat([[{"role": "user", "content": "What is 17*23? End with #### answer."}]],
               SamplingParams(temperature=0, max_tokens=200))
print(out[0].outputs[0].text)

(MainProcess pid=15471) INFO 11-16 21:18:12 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
(MainProcess pid=15471) INFO 11-16 21:18:13 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=15471) WARNING 11-16 21:18:13 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=15471) INFO 11-16 21:18:13 [model.py:2170] Using max model len 2048
(MainProcess pid=15471) INFO 11-16 21:18:13 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  7.38it/s, est. speed input: 320.38 toks/s, output: 37.25 toks/s]

491####


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ha_home
%cd /content
!rm -rf /content/ha
!git clone https://github.com/Soumya080/harness-admission-colab.git /content/ha
%cd /content/ha
!ls

In [26]:
# CPU checks: must print 'passed' and 'ALL DONE'
!python -m pytest -q tests
!HA_FAKE=1 HA_HOME=$(mktemp -d) bash scripts/run_all.sh 2>&1 | tail -2

....................                                                     [100%]
20 passed in 1.26s

ALL DONE. Log: logs/run_20261116_2119.log   Results: /tmp/tmp.7SGVG46aYf/results


In [27]:
import os, sys
os.chdir('/content/ha')
sys.path.insert(0, '/content/ha')
os.environ['HA_HOME'] = '/content/drive/MyDrive/ha_home'   # or '/content/ha_home' if you're not using Drive
os.environ['HA_DTYPE'] = 'half'
os.environ['HA_GPU_UTIL'] = '0.85'

from harness.store import list_harnesses, load_harness
tags = [load_harness(x).tag for x in list_harnesses()]
print({t: tags.count(t) for t in set(tags)})      # expect base:1, planted:1, noop:5 (and llm:N if any)
n = tags.count('llm')
if n < 30:
    !python -m proposer.propose --n {30 - n}

{'noop': 5, 'planted': 1, 'base': 1}
79 proposal-pool failures
W1116 21:23:50.255000 31863 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1116 21:23:50.255000 31863 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1116 21:23:50.256000 31863 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=31863) INFO 11-16 21:23:58 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
config.json: 100% 841/841 [00:00<00:00, 4.98MB/s]
(MainProcess pid=31863) INFO 11-16 21:23:59 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=31863) INFO 11-16 21:23:59 [model.py:2170] Using max model len 4096
(MainProcess pid=31863) INFO 11-16 21:24:01 [scheduler.py:310] Chunked prefill is enabled with max_num_bat

In [ ]:
import os, sys
os.chdir('/content/ha')
sys.path.insert(0, '/content/ha')
os.environ['HA_HOME'] = '/content/drive/MyDrive/ha_home'   # or '/content/ha_home' if you're not using Drive
os.environ['HA_DTYPE'] = 'half'
os.environ['HA_GPU_UTIL'] = '0.85'

from harness.store import list_harnesses, load_harness
tags = [load_harness(x).tag for x in list_harnesses()]
print({t: tags.count(t) for t in set(tags)})      # expect base:1, planted:1, noop:5 (and llm:N if any)
n = tags.count('llm')
if n < 30:
    !python -m proposer.propose --n {30 - n}

**Only continue if the cell above printed `CHECK PASSED`.** Otherwise send the output.

In [29]:
# Proposer (7B AWQ). Restart runtime first if you hit OOM, then rerun cells 2-3 and this one.
from harness.store import list_harnesses, load_harness
n = sum(load_harness(x).tag == 'llm' for x in list_harnesses())
print('llm edits so far:', n)
if n < 30:
    !python -m proposer.propose --n {30 - n}

llm edits so far: 0
79 proposal-pool failures
W1116 21:24:48.088000 32435 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1116 21:24:48.089000 32435 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1116 21:24:48.089000 32435 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=32435) INFO 11-16 21:24:55 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
(MainProcess pid=32435) INFO 11-16 21:24:57 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=32435) INFO 11-16 21:24:57 [model.py:2170] Using max model len 4096
(MainProcess pid=32435) INFO 11-16 21:24:59 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.
Parse safetensors files: 100% 2/2 [00:00<00:00,  

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
!nvidia-smi
!ps aux | grep -E "vllm|EngineCore|python -m" | grep -v grep

In [ ]:
!pkill -9 -f EngineCore; pkill -9 -f "python -m scripts"; pkill -9 -f "python -m proposer"; sleep 3
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
# Full grid: the long step (~1-3 h). Rerun this cell after any disconnect.
!python -m scripts.run_grid

In [ ]:
!rm -f $HA_HOME/cache/outcomes/*.parquet
!python -m scripts.run_grid --only base
!python -m scripts.run_grid --only planted
!python -m scripts.check_stop
!python -m scripts.run_grid

In [ ]:
# Quick diagnostics on Colab (CPU steps for the report are better on your laptop)
!python -m scripts.diagnose | tee $HA_HOME/results_diagnose.txt

## Then on your laptop
Download `MyDrive/ha_home`, then:
```bash
export HA_HOME=/path/to/ha_home
python -m scripts.diagnose | tee $HA_HOME/results/diagnose.txt
python -m sims.stage0 | tee $HA_HOME/results/stage0.txt
python -m sims.plot_stage0
python -m scripts.make_report --runs 1000 --ablations
```